## mvp.bronze.pix_transacoes

In [0]:
%sql
-- ==============================================================================
-- 1. INFRAESTRUTURA E GOVERNANÇA DO SCHEMA BRONZE
-- ==============================================================================
CREATE SCHEMA IF NOT EXISTS mvp.bronze
COMMENT 'Camada Bronze: dados brutos tipados e auditados originados dos Volumes de Staging';

-- ==============================================================================
-- 2. DDL DA TABELA DELTA BRONZE (Criação Idempotente)
-- ==============================================================================
CREATE TABLE IF NOT EXISTS mvp.bronze.pix_transacoes (
    anomes                                  INT,
    pag_pfpj                                STRING,
    rec_pfpj                                STRING,
    pag_regiao                              STRING,
    rec_regiao                              STRING,
    pag_idade                               STRING,
    rec_idade                               STRING,
    forma_iniciacao                         STRING,
    natureza                                STRING,
    finalidade                              STRING,
    montante_reais                          DECIMAL(18, 2),
    quantidade_tx                           BIGINT,
    _input_file_name                        STRING,
    _file_path                              STRING,
    _file_size_bytes                        BIGINT,
    _file_modification_time                 TIMESTAMP,
    _ingestion_timestamp                    TIMESTAMP
)
PARTITIONED BY (anomes)
COMMENT 'Tabela Bronze contendo microdados agregados de transações Pix do Banco Central alimentada incrementalmente';

-- ==============================================================================
-- 3. VIEW TEMPORÁRIA COM DEDUPLICAÇÃO TÉCNICA NA ORIGEM (Evita SQLSTATE 21506)
-- ==============================================================================
CREATE OR REPLACE TEMPORARY VIEW vw_staging_pix_transacoes AS
SELECT 
    CAST(AnoMes AS INT)                                              AS anomes,
    UPPER(TRIM(PAG_PFPJ))                                           AS pag_pfpj,
    UPPER(TRIM(REC_PFPJ))                                           AS rec_pfpj,
    UPPER(TRIM(PAG_REGIAO))                                         AS pag_regiao,
    UPPER(TRIM(REC_REGIAO))                                         AS rec_regiao,
    TRIM(PAG_IDADE)                                                 AS pag_idade,
    TRIM(REC_IDADE)                                                 AS rec_idade,
    UPPER(TRIM(FORMAINICIACAO))                                     AS forma_iniciacao,
    UPPER(TRIM(NATUREZA))                                           AS natureza,
    UPPER(TRIM(FINALIDADE))                                         AS finalidade,
    CAST(REPLACE(CAST(VALOR AS STRING), ',', '.') AS DECIMAL(18, 2)) AS montante_reais,
    CAST(QUANTIDADE AS BIGINT)                                      AS quantidade_tx,
    -- Colunas de metadados
    _metadata.file_name                                             AS _input_file_name,
    _metadata.file_path                                             AS _file_path,
    _metadata.file_size                                             AS _file_size_bytes,
    _metadata.file_modification_time                                AS _file_modification_time,
    current_timestamp()                                             AS _ingestion_timestamp
FROM read_files(
    '/Volumes/mvp/staging/pix_transacoes/*.json',
    format => 'json',
    multiline => true
)
-- Garante que cada combinação na fonte seja estritamente única, pegando o arquivo mais recente
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY 
        anomes,
        pag_pfpj,
        rec_pfpj,
        pag_regiao,
        rec_regiao,
        pag_idade,
        rec_idade,
        forma_iniciacao,
        natureza,
        finalidade
    ORDER BY _file_modification_time DESC
) = 1;

-- ==============================================================================
-- 4. CARGA INCREMENTAL COM MERGE INTO
-- ==============================================================================
MERGE INTO mvp.bronze.pix_transacoes AS target
USING vw_staging_pix_transacoes AS source
ON target.anomes = source.anomes
   AND target.pag_pfpj <=> source.pag_pfpj
   AND target.rec_pfpj <=> source.rec_pfpj
   AND target.pag_regiao <=> source.pag_regiao
   AND target.rec_regiao <=> source.rec_regiao
   AND target.pag_idade <=> source.pag_idade
   AND target.rec_idade <=> source.rec_idade
   AND target.forma_iniciacao <=> source.forma_iniciacao
   AND target.natureza <=> source.natureza
   AND target.finalidade <=> source.finalidade
WHEN MATCHED AND target._file_modification_time < source._file_modification_time THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

-- ==============================================================================
-- 5. AUDITORIA E RESUMO TÉCNICO
-- ==============================================================================
SELECT 
    anomes,
    COUNT(1)                                    AS total_linhas,
    SUM(quantidade_tx)                          AS volume_transacoes,
    SUM(montante_reais)                         AS valor_total_reais,
    COUNT(DISTINCT _input_file_name)            AS qtd_arquivos_origem,
    SUM(_file_size_bytes)                       AS tamanho_total_bytes,
    MIN(_file_modification_time)                AS min_modificacao_arquivo,
    MAX(_file_modification_time)                AS max_modificacao_arquivo,
    MAX(_ingestion_timestamp)                   AS ultimo_processamento
FROM mvp.bronze.pix_transacoes
GROUP BY anomes
ORDER BY anomes;

anomes,total_linhas,volume_transacoes,valor_total_reais,qtd_arquivos_origem,tamanho_total_bytes,min_modificacao_arquivo,max_modificacao_arquivo,ultimo_processamento
202011,19,71010,53890784.93,1,53647640,2026-09-07T14:20:09.000Z,2026-09-07T14:20:09.000Z,2026-09-21T22:33:50.521Z
202012,58,1048882,6587200214.80,1,163746006,2026-09-07T14:20:11.000Z,2026-09-07T14:20:11.000Z,2026-09-21T22:54:16.065Z
202101,90,5406459,2451174851.01,1,254046330,2026-09-07T14:20:12.000Z,2026-09-07T14:20:12.000Z,2026-09-21T22:54:16.065Z
202102,138,2105482,1421855131.62,1,389450628,2026-09-07T14:20:14.000Z,2026-09-07T14:20:14.000Z,2026-09-21T22:54:16.065Z
202103,195,31885595,21164091556.05,1,550219605,2026-09-07T14:20:16.000Z,2026-09-07T14:20:16.000Z,2026-09-21T22:54:16.065Z
202104,132,9915906,2661116793.54,1,372365136,2026-09-07T14:20:36.000Z,2026-09-07T14:20:36.000Z,2026-09-21T22:54:16.065Z
202105,258,87409138,23751843986.07,1,727676358,2026-09-07T14:20:38.000Z,2026-09-07T14:20:38.000Z,2026-09-21T22:54:16.065Z
202106,1065,42427312,15023487801.56,1,3003109365,2026-09-07T14:20:39.000Z,2026-09-07T14:20:39.000Z,2026-09-21T22:54:16.065Z
202107,1142,125711405,44687451057.09,1,3219903260,2026-09-07T14:20:41.000Z,2026-09-07T14:20:41.000Z,2026-09-21T22:54:16.065Z
202108,1278,87280444,32156679175.73,1,3603218760,2026-09-07T14:20:43.000Z,2026-09-07T14:20:43.000Z,2026-09-21T22:54:16.065Z


## mvp.bronze.pix_fraudes_med

In [0]:
%sql
-- ==============================================================================
-- 1. INFRAESTRUTURA E GOVERNANÇA DO SCHEMA BRONZE
-- ==============================================================================
CREATE SCHEMA IF NOT EXISTS mvp.bronze
COMMENT 'Camada Bronze: dados brutos tipados e auditados originados dos Volumes de Staging';

-- ==============================================================================
-- 2. DDL DA TABELA DELTA BRONZE (Criação Idempotente)
-- ==============================================================================
CREATE TABLE IF NOT EXISTS mvp.bronze.pix_fraudes_med (
    anomes                                  INT,
    qtde_pix_contestados                    BIGINT,
    qtde_contestacoes_aceitas               BIGINT,
    qtde_contestacoes_rejeitadas            BIGINT,
    taxa_aceite_por_100k                    DECIMAL(12, 4),
    qtde_usuarios_marcados_fraude           BIGINT,
    qtde_chaves_marcadas_fraude             BIGINT,
    valor_contestacoes_aceitas_reais        DECIMAL(18, 2),
    qtde_devolucoes_integrais               BIGINT,
    valor_devolvido_integral_reais          DECIMAL(18, 2),
    qtde_devolucoes_parciais                BIGINT,
    valor_devolvido_parcial_reais           DECIMAL(18, 2),
    valor_residual_nao_devolvido_reais      DECIMAL(18, 2),
    qtde_nao_devolvida_saldo_insuficiente   BIGINT,
    valor_nao_devolvido_saldo_insuficiente  DECIMAL(18, 2),
    qtde_nao_devolvida_conta_encerrada      BIGINT,
    valor_nao_devolvido_conta_encerrada     DECIMAL(18, 2),
    qtde_nao_devolvida_motivos_diversos     BIGINT,
    valor_nao_devolvido_motivos_diversos    DECIMAL(18, 2),
    percentual_devolucao                    DECIMAL(6, 2),
    qtde_bloqueio_cautelar_liberados        BIGINT,
    valor_bloqueio_cautelar_liberado_reais  DECIMAL(18, 2),
    qtde_bloqueio_cautelar_devolvidos       BIGINT,
    valor_bloqueio_cautelar_devolvido_reais DECIMAL(18, 2),
    -- Metadados de linhagem e rastreabilidade técnica
    _input_file_name                        STRING,
    _file_path                              STRING,
    _file_size_bytes                        BIGINT,
    _file_modification_time                 TIMESTAMP,
    _ingestion_timestamp                    TIMESTAMP
)
PARTITIONED BY (anomes)
COMMENT 'Tabela Bronze contendo microdados agregados de fraudes e devoluções MED do Banco Central';

-- ==============================================================================
-- 3. VIEW TEMPORÁRIA COM DEDUPLICAÇÃO NA ORIGEM (Proteção contra SQLSTATE 21506)
-- ==============================================================================
CREATE OR REPLACE TEMPORARY VIEW vw_staging_pix_fraudes_med AS
SELECT 
    CAST(AnoMes AS INT)                                                                    AS anomes,
    CAST(QtdePixcontestados AS BIGINT)                                                     AS qtde_pix_contestados,
    CAST(Qtdecontestacoesaceitas AS BIGINT)                                                AS qtde_contestacoes_aceitas,
    CAST(Qtdecontestacoesrejeitadas AS BIGINT)                                             AS qtde_contestacoes_rejeitadas,
    CAST(REPLACE(CAST(Qtdecontestacoesaceitasacada100mil AS STRING), ',', '.') AS DECIMAL(12, 4)) AS taxa_aceite_por_100k,
    CAST(QtdeUsuarioscommarcacoesdefraude AS BIGINT)                                       AS qtde_usuarios_marcados_fraude,
    CAST(QtdeChavesPixcommarcacoesdefraude AS BIGINT)                                      AS qtde_chaves_marcadas_fraude,
    CAST(REPLACE(CAST(ValorPixcontestadosaceitos AS STRING), ',', '.') AS DECIMAL(18, 2))  AS valor_contestacoes_aceitas_reais,
    CAST(QuantidadedevolvidaintegralmentepormeiodoMED AS BIGINT)                           AS qtde_devolucoes_integrais,
    CAST(REPLACE(CAST(ValorPixdevolvidosintegralmente AS STRING), ',', '.') AS DECIMAL(18, 2)) AS valor_devolvido_integral_reais,
    CAST(QuantidadedevolvidaparcialmentepormeiodoMED AS BIGINT)                            AS qtde_devolucoes_parciais,
    CAST(REPLACE(CAST(ValorPixdevolvidosparcialmente AS STRING), ',', '.') AS DECIMAL(18, 2)) AS valor_devolvido_parcial_reais,
    CAST(REPLACE(CAST(ValorPixresidualnaodevolvido AS STRING), ',', '.') AS DECIMAL(18, 2)) AS valor_residual_nao_devolvido_reais,
    CAST(Quantidadedenaodevolvidossaldoinsuficiente AS BIGINT)                             AS qtde_nao_devolvida_saldo_insuficiente,
    CAST(REPLACE(CAST(ValorPixnaodevolvidossaldoinsuficiente AS STRING), ',', '.') AS DECIMAL(18, 2)) AS valor_nao_devolvido_saldo_insuficiente,
    CAST(Quantidadedenaodevolvidoscontaencerrada AS BIGINT)                                AS qtde_nao_devolvida_conta_encerrada,
    CAST(REPLACE(CAST(Valornaodevolvidoscontaencerrada AS STRING), ',', '.') AS DECIMAL(18, 2)) AS valor_nao_devolvido_conta_encerrada,
    CAST(Quantidadedenaodevolvidosmotivosdiversos AS BIGINT)                               AS qtde_nao_devolvida_motivos_diversos,
    CAST(REPLACE(CAST(ValorPixnaodevolvidosmotivosdiversos AS STRING), ',', '.') AS DECIMAL(18, 2)) AS valor_nao_devolvido_motivos_diversos,
    CAST(REPLACE(CAST(PercentualdeDevolucao AS STRING), ',', '.') AS DECIMAL(6, 2))        AS percentual_devolucao,
    CAST(QtdePixbloqueadoscautelarmenteeliberados AS BIGINT)                               AS qtde_bloqueio_cautelar_liberados,
    CAST(REPLACE(CAST(ValorPixbloqueadoscautelarmenteeliberados AS STRING), ',', '.') AS DECIMAL(18, 2)) AS valor_bloqueio_cautelar_liberado_reais,
    CAST(QtdePixbloqueadoscautelarmenteedevolvidos AS BIGINT)                              AS qtde_bloqueio_cautelar_devolvidos,
    CAST(REPLACE(CAST(ValorPixbloqueadoscautelarmenteedevolvidos AS STRING), ',', '.') AS DECIMAL(18, 2)) AS valor_bloqueio_cautelar_devolvido_reais,
    -- Linhagem e metadados
    _metadata.file_name                                                                    AS _input_file_name,
    _metadata.file_path                                                                    AS _file_path,
    _metadata.file_size                                                                    AS _file_size_bytes,
    _metadata.file_modification_time                                                       AS _file_modification_time,
    current_timestamp()                                                                    AS _ingestion_timestamp
FROM read_files(
    '/Volumes/mvp/staging/pix_fraudes_med/*.json',
    format => 'json',
    multiline => true
)
-- Garante apenas 1 registro por anomes na leitura caso múltiplos arquivos cubram o mesmo mês
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY anomes 
    ORDER BY _metadata.file_modification_time DESC
) = 1;

-- ==============================================================================
-- 4. CARGA INCREMENTAL COM MERGE INTO (IDEMPOTÊNCIA E PODA DE PARTIÇÃO)
-- ==============================================================================
MERGE INTO mvp.bronze.pix_fraudes_med AS target
USING vw_staging_pix_fraudes_med AS source
ON target.anomes = source.anomes
WHEN MATCHED AND target._file_modification_time < source._file_modification_time THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

-- ==============================================================================
-- 5. AUDITORIA TÉCNICA E RESUMO DAS COMPETÊNCIAS
-- ==============================================================================
SELECT 
    anomes,
    COUNT(1)                                    AS total_linhas,
    SUM(qtde_pix_contestados)                   AS total_contestacoes,
    SUM(qtde_contestacoes_aceitas)              AS contestacoes_aceitas,
    SUM(qtde_usuarios_marcados_fraude)          AS usuarios_marcados_fraude,
    SUM(valor_contestacoes_aceitas_reais)       AS montante_contestado_reais,
    SUM(valor_devolvido_integral_reais)         AS montante_devolvido_reais,
    COUNT(DISTINCT _input_file_name)            AS qtd_arquivos_origem,
    SUM(_file_size_bytes)                       AS tamanho_total_arquivos_bytes,
    MIN(_file_modification_time)                AS min_modificacao_arquivo,
    MAX(_file_modification_time)                AS max_modificacao_arquivo,
    MAX(_ingestion_timestamp)                   AS ultimo_processamento
FROM mvp.bronze.pix_fraudes_med
GROUP BY anomes
ORDER BY anomes;

anomes,total_linhas,total_contestacoes,contestacoes_aceitas,usuarios_marcados_fraude,montante_contestado_reais,montante_devolvido_reais,qtd_arquivos_origem,tamanho_total_arquivos_bytes,min_modificacao_arquivo,max_modificacao_arquivo,ultimo_processamento
202201,3,214524,173331,67527,367676604.84,12785286.30,1,173745,2026-09-07T13:54:09.000Z,2026-09-07T13:54:09.000Z,2026-09-21T22:56:58.695Z
202202,4,314544,242720,95940,544136634.32,21421092.24,1,227260,2026-09-07T13:54:10.000Z,2026-09-07T13:54:10.000Z,2026-09-21T22:56:58.695Z
202203,5,674820,525080,177965,1084699665.60,54990460.30,1,278580,2026-09-07T13:54:44.000Z,2026-09-07T13:54:44.000Z,2026-09-21T22:56:58.695Z
202204,6,915816,688578,194142,1068982412.04,50967381.90,1,327648,2026-09-07T13:55:19.000Z,2026-09-07T13:55:19.000Z,2026-09-21T22:56:58.695Z
202205,7,1605807,1043042,248178,1574040528.83,57259573.00,1,374521,2026-09-07T13:55:54.000Z,2026-09-07T13:55:54.000Z,2026-09-21T22:56:58.695Z
202206,8,1521648,1172208,305024,1789420878.96,120274583.76,1,419184,2026-09-07T13:55:55.000Z,2026-09-07T13:55:55.000Z,2026-09-21T22:56:58.695Z
202207,9,1674504,1282437,348678,1923435487.62,88139944.35,1,461610,2026-09-07T13:56:27.000Z,2026-09-07T13:56:27.000Z,2026-09-21T22:56:58.695Z
202208,10,2181310,1550210,417370,2339078237.50,107590535.50,1,501830,2026-09-07T13:56:28.000Z,2026-09-07T13:56:28.000Z,2026-09-21T22:56:58.695Z
202209,11,2160147,1515052,449999,2429067540.90,141159166.72,1,539847,2026-09-07T13:57:01.000Z,2026-09-07T13:57:01.000Z,2026-09-21T22:56:58.695Z
202210,12,2392368,1765356,510132,2786288895.24,186273638.04,1,575640,2026-09-07T13:57:34.000Z,2026-09-07T13:57:34.000Z,2026-09-21T22:56:58.695Z


## mvp.bronze.ibge_populacao

In [0]:
%sql
-- ==============================================================================
-- 1. INFRAESTRUTURA E GOVERNANÇA DO SCHEMA BRONZE
-- ==============================================================================
CREATE SCHEMA IF NOT EXISTS mvp.bronze
COMMENT 'Camada Bronze: dados brutos tipados e auditados originados dos Volumes de Staging';

-- ==============================================================================
-- 2. DDL DA TABELA DELTA BRONZE (Criação Idempotente sem Particionamento)
-- ==============================================================================
CREATE TABLE IF NOT EXISTS mvp.bronze.ibge_populacao (
    id_regiao_ibge                          INT,
    nome_regiao                             STRING,
    ano_referencia                          INT,
    populacao_residente                     BIGINT,
    -- Linhagem técnica e metadados de rastreabilidade
    _input_file_name                        STRING,
    _file_path                              STRING,
    _file_size_bytes                        BIGINT,
    _file_modification_time                 TIMESTAMP,
    _ingestion_timestamp                    TIMESTAMP
)
COMMENT 'Tabela Bronze contendo a população residente por Grande Região (IBGE SIDRA Tabela 4714) alimentada incrementalmente';

-- ==============================================================================
-- 3. VIEW TEMPORÁRIA COM DEDUPLICAÇÃO TÉCNICA NA ORIGEM (Evita SQLSTATE 21506)
-- ==============================================================================
CREATE OR REPLACE TEMPORARY VIEW vw_staging_ibge_populacao AS
SELECT 
    CAST(D1C AS INT)                                   AS id_regiao_ibge,
    UPPER(TRIM(D1N))                                   AS nome_regiao,
    CAST(D3N AS INT)                                   AS ano_referencia,
    CAST(V AS BIGINT)                                  AS populacao_residente,
    -- Colunas de metadados técnicos
    _metadata.file_name                                AS _input_file_name,
    _metadata.file_path                                AS _file_path,
    _metadata.file_size                                AS _file_size_bytes,
    _metadata.file_modification_time                   AS _file_modification_time,
    current_timestamp()                                AS _ingestion_timestamp
FROM read_files(
    '/Volumes/mvp/staging/ibge_populacao/*.json',
    format => 'json',
    multiline => true
)
-- Garante apenas 1 registro por região e ano caso existam múltiplos arquivos no volume
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY D1C, D3N 
    ORDER BY _metadata.file_modification_time DESC
) = 1;

-- ==============================================================================
-- 4. CARGA INCREMENTAL COM MERGE INTO (IDEMPOTENTE)
-- ==============================================================================
MERGE INTO mvp.bronze.ibge_populacao AS target
USING vw_staging_ibge_populacao AS source
ON target.id_regiao_ibge = source.id_regiao_ibge
   AND target.ano_referencia = source.ano_referencia
WHEN MATCHED AND target._file_modification_time < source._file_modification_time THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

-- ==============================================================================
-- 5. AUDITORIA E VALIDAÇÃO DOS DADOS DEMOGRÁFICOS
-- ==============================================================================
SELECT 
    id_regiao_ibge,
    nome_regiao,
    ano_referencia,
    populacao_residente,
    _input_file_name,
    _file_size_bytes,
    _file_modification_time,
    _ingestion_timestamp
FROM mvp.bronze.ibge_populacao
ORDER BY id_regiao_ibge;

id_regiao_ibge,nome_regiao,ano_referencia,populacao_residente,_input_file_name,_file_size_bytes,_file_modification_time,_ingestion_timestamp
1,NORTE,2022,17354884,ibge_populacao_regioes.json,940,2026-09-07T14:24:25.000Z,2026-09-07T15:10:14.684Z
2,NORDESTE,2022,54658515,ibge_populacao_regioes.json,940,2026-09-07T14:24:25.000Z,2026-09-07T15:10:14.684Z
3,SUDESTE,2022,84840113,ibge_populacao_regioes.json,940,2026-09-07T14:24:25.000Z,2026-09-07T15:10:14.684Z
4,SUL,2022,29937706,ibge_populacao_regioes.json,940,2026-09-07T14:24:25.000Z,2026-09-07T15:10:14.684Z
5,CENTRO-OESTE,2022,16289538,ibge_populacao_regioes.json,940,2026-09-07T14:24:25.000Z,2026-09-07T15:10:14.684Z


In [0]:
%sql
-- ==============================================================================
-- AUDITORIA CONSOLIDADA DE METADADOS TÉCNICOS: CAMADA BRONZE
-- Consolida volumetria, arquivos processados, cobertura temporal e timestamps
-- ==============================================================================
WITH auditoria_bronze AS (
    -- 1. Transações Pix
    SELECT 
        'mvp.bronze.pix_transacoes'             AS tabela_bronze,
        'Temporal / Transacional'               AS tipo_carga,
        COUNT(1)                                AS total_registros,
        COUNT(DISTINCT anomes)                  AS total_particoes_periodos,
        CAST(MIN(anomes) AS STRING)             AS primeiro_periodo,
        CAST(MAX(anomes) AS STRING)             AS ultimo_periodo,
        COUNT(DISTINCT _input_file_name)        AS total_arquivos_origem,
        ROUND(SUM(_file_size_bytes) / 1024 / 1024, 2) AS tamanho_total_arquivos_mb,
        MIN(_file_modification_time)            AS min_modificacao_origem,
        MAX(_file_modification_time)            AS max_modificacao_origem,
        MIN(_ingestion_timestamp)               AS primeiro_processamento,
        MAX(_ingestion_timestamp)               AS ultimo_processamento
    FROM mvp.bronze.pix_transacoes

    UNION ALL

    -- 2. Fraudes MED
    SELECT 
        'mvp.bronze.pix_fraudes_med'            AS tabela_bronze,
        'Temporal / Transacional'               AS tipo_carga,
        COUNT(1)                                AS total_registros,
        COUNT(DISTINCT anomes)                  AS total_particoes_periodos,
        CAST(MIN(anomes) AS STRING)             AS primeiro_periodo,
        CAST(MAX(anomes) AS STRING)             AS ultimo_periodo,
        COUNT(DISTINCT _input_file_name)        AS total_arquivos_origem,
        ROUND(SUM(_file_size_bytes) / 1024 / 1024, 2) AS tamanho_total_arquivos_mb,
        MIN(_file_modification_time)            AS min_modificacao_origem,
        MAX(_file_modification_time)            AS max_modificacao_origem,
        MIN(_ingestion_timestamp)               AS primeiro_processamento,
        MAX(_ingestion_timestamp)               AS ultimo_processamento
    FROM mvp.bronze.pix_fraudes_med

    UNION ALL

    -- 3. População IBGE
    SELECT 
        'mvp.bronze.ibge_populacao'             AS tabela_bronze,
        'Referência / Cadastral'                AS tipo_carga,
        COUNT(1)                                AS total_registros,
        COUNT(DISTINCT id_regiao_ibge)          AS total_particoes_periodos,
        CAST(MIN(ano_referencia) AS STRING)     AS primeiro_periodo,
        CAST(MAX(ano_referencia) AS STRING)     AS ultimo_periodo,
        COUNT(DISTINCT _input_file_name)        AS total_arquivos_origem,
        ROUND(SUM(_file_size_bytes) / 1024 / 1024, 2) AS tamanho_total_arquivos_mb,
        MIN(_file_modification_time)            AS min_modificacao_origem,
        MAX(_file_modification_time)            AS max_modificacao_origem,
        MIN(_ingestion_timestamp)               AS primeiro_processamento,
        MAX(_ingestion_timestamp)               AS ultimo_processamento
    FROM mvp.bronze.ibge_populacao
)
SELECT 
    tabela_bronze,
    tipo_carga,
    total_registros,
    total_particoes_periodos,
    primeiro_periodo,
    ultimo_periodo,
    total_arquivos_origem,
    tamanho_total_arquivos_mb,
    min_modificacao_origem,
    max_modificacao_origem,
    primeiro_processamento,
    ultimo_processamento
FROM auditoria_bronze;

tabela_bronze,tipo_carga,total_registros,total_particoes_periodos,primeiro_periodo,ultimo_periodo,total_arquivos_origem,tamanho_total_arquivos_mb,min_modificacao_origem,max_modificacao_origem,primeiro_processamento,ultimo_processamento
mvp.bronze.pix_transacoes,Temporal / Transacional,700000,70,202011,202608,70,1880404.15,2026-09-07T14:20:09.000Z,2026-09-07T14:24:22.000Z,2026-09-21T22:33:50.521Z,2026-09-21T22:54:16.065Z
mvp.bronze.pix_fraudes_med,Temporal / Transacional,1456,52,202201,202604,51,29.1,2026-09-07T13:54:09.000Z,2026-09-07T14:15:35.000Z,2026-09-07T15:18:43.830Z,2026-09-21T22:56:58.695Z
mvp.bronze.ibge_populacao,Referência / Cadastral,5,5,2022,2022,1,0.0,2026-09-07T14:24:25.000Z,2026-09-07T14:24:25.000Z,2026-09-07T15:10:14.684Z,2026-09-07T15:10:14.684Z
